In [1]:
import os
import pandas as pd
from uuid import uuid4

from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

In [2]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

index = pc.Index("pinecone-data")

In [4]:
import os

print(os.getcwd())
print(os.listdir())

c:\Learnership\Python projects\AI-Application\RAG-chatbot-pinecone
['.env', '.gitignore', 'Day1.ipynb', 'README.md', 'upserting YouTube Transcripts.ipynb']


In [5]:
youtube_df = pd.read_csv("youtube_rag_data_small.csv")

youtube_df.head()

,id,blob,channel_id,end,published,start,text,title,url
0,35Pdoyi6ZoQ-t0.0,"{'channel_id': 'UCv83tO5cePwHMt1952IVVHw', 'en...",UCv83tO5cePwHMt1952IVVHw,74,2021-07-06 13:00:03 UTC,0,"Hi, welcome to the video. So this is the fourt...",Training and Testing an Italian BERT - Transfo...,https://youtu.be/35Pdoyi6ZoQ
1,35Pdoyi6ZoQ-t18.48,"{'channel_id': 'UCv83tO5cePwHMt1952IVVHw', 'en...",UCv83tO5cePwHMt1952IVVHw,94,2021-07-06 13:00:03 UTC,18,So we got some data. We built a tokenizer with...,Training and Testing an Italian BERT - Transfo...,https://youtu.be/35Pdoyi6ZoQ
2,35Pdoyi6ZoQ-t32.36,"{'channel_id': 'UCv83tO5cePwHMt1952IVVHw', 'en...",UCv83tO5cePwHMt1952IVVHw,108,2021-07-06 13:00:03 UTC,32,So let's move over to the code. And we see her...,Training and Testing an Italian BERT - Transfo...,https://youtu.be/35Pdoyi6ZoQ
3,35Pdoyi6ZoQ-t51.519999999999996,"{'channel_id': 'UCv83tO5cePwHMt1952IVVHw', 'en...",UCv83tO5cePwHMt1952IVVHw,125,2021-07-06 13:00:03 UTC,51,"PyTorch data loader, ready. And we can begin t...",Training and Testing an Italian BERT - Transfo...,https://youtu.be/35Pdoyi6ZoQ
4,35Pdoyi6ZoQ-t67.28,"{'channel_id': 'UCv83tO5cePwHMt1952IVVHw', 'en...",UCv83tO5cePwHMt1952IVVHw,140,2021-07-06 13:00:03 UTC,67,So when we're training a model for mass langua...,Training and Testing an Italian BERT - Transfo...,https://youtu.be/35Pdoyi6ZoQ


In [6]:
batch_limit = 100

count = 0

In [7]:
for i in range(0, len(youtube_df), batch_limit):

    batch = youtube_df.iloc[i:i+batch_limit]

    metadatas = [
        {
            "text_id": row["id"],
            "text": row["text"],
            "title": row["title"],
            "url": row["url"],
            "published": row["published"]
        }
        for _, row in batch.iterrows()
    ]

    texts = batch["text"].tolist()

    ids = [str(uuid4()) for _ in range(len(texts))]

    response = client.embeddings.create(
        input=texts,
        model="text-embedding-3-small"
    )

    embeds = [list(record.embedding) for record in response.data]

    vectors = list(zip(ids, embeds, metadatas))

    index.upsert(
        vectors=vectors,
        namespace="youtube_rag_dataset"
    )

    count += 1

    print(f"Batch {count} uploaded")

Batch 1 uploaded
Batch 2 uploaded
Batch 3 uploaded
Batch 4 uploaded
Batch 5 uploaded
Batch 6 uploaded
Batch 7 uploaded
Batch 8 uploaded
Batch 9 uploaded
Batch 10 uploaded
Batch 11 uploaded


In [8]:
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=1142, metric='cosine', namespaces=2)
